In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

In [ ]:
# OPTION 1 - BACKWARD STRATEGY (BS)
def backward_strategy_prune(model, target_layer_index):
    """
    Removes a specific hidden layer and reconfigures weights to correctly connect
    adjacent layers. Redirects weights from the pruned layer to the next layer,
    adjusting their size if necessary.
    """
    
    layers = model.layers

    if target_layer_index < 2 or target_layer_index >= len(layers) - 1:
        raise ValueError("The hidden layer index must be between 2 and the total number of hidden layers.")

    print("\n=== STARTING PRUNING PROCESS ===")
    print(f"Target layer index to remove: {target_layer_index}")
    print(f"Total layers in model: {len(layers)}")

    # Involved layers
    previous_layer = layers[target_layer_index - 1]
    layer_to_prune = layers[target_layer_index]
    next_layer = layers[target_layer_index + 1]

    print("\n--- Layers Involved ---")
    print(f"Previous Layer: {previous_layer.name}")
    print(f"Layer to Prune: {layer_to_prune.name}")
    print(f"Next Layer: {next_layer.name}")

    print(f"\nRetrieving weights and biases from target layer: {layer_to_prune.name}")
    print(f"Using '{previous_layer.name}' as reference layer.")

    # Retrieve weights and biases
    input_weights, input_bias = layer_to_prune.get_weights()
    print(f"\nTarget layer weights (shape {input_weights.shape}):\n{input_weights}")
    print(f"Target layer bias:\n{input_bias}")

    ref_weights, ref_bias = previous_layer.get_weights()
    print(f"\nReference layer weights (shape {ref_weights.shape}):\n{ref_weights}")
    print(f"Reference layer bias:\n{ref_bias}")

    num_neurons_prev = input_weights.shape[0]
    num_neurons_prune = input_weights.shape[1]
    num_neurons_next = next_layer.units

    print("\n--- Weight Adjustment ---")
    print(f"Neurons in previous layer: {num_neurons_prev}")
    print(f"Neurons in layer to prune: {num_neurons_prune}")
    print(f"Neurons in next layer: {num_neurons_next}")

    if input_weights.shape[1] > num_neurons_next:
        print("\n------------------------- OPTION 1, BS ----------------------")
        print("\nReducing weights and bias: selecting neurons based on absolute mean similarity.")

        # 1. Target layer means
        origin_means = np.mean(input_weights, axis=0)
        print("\nMean vector (target layer):")
        print(np.round(origin_means, 4))

        # 2. Next layer means
        next_weights, _ = next_layer.get_weights()
        reference_means = np.mean(next_weights, axis=0)
        print("\nMean vector (next layer):")
        print(np.round(reference_means, 4))

        # 3. Select indices of the closest neurons (without repetition)
        selected_indices = []
        used_indices = set()
        for ref_val in reference_means:
            differences = np.abs(origin_means - ref_val)
            sorted_order = np.argsort(differences)
            for idx in sorted_order:
                if idx not in used_indices:
                    selected_indices.append(idx)
                    used_indices.add(idx)
                    break

        print("\nFinal selected indices based on mean similarity:")
        print(selected_indices)

        # 4. Filter weights and biases using selected indices
        input_weights = input_weights[:, selected_indices]
        input_bias = input_bias[selected_indices]

    elif input_weights.shape[1] < num_neurons_next:
        print("\n------- Next layer neuron count is larger; BS not applicable --------")
        return model

    print(f"New weights shape for connecting previous to next layer: {input_weights.shape}")
    print(f"New adjusted bias shape: {input_bias.shape}")

    new_weights = input_weights
    new_bias = input_bias

    # FIXED FOR Keras 3/TF 2.16.1
    # 1. Get model input shape
    full_input_shape = model.input_shape
    
    # 2. Handle multiple inputs (if list)
    if isinstance(full_input_shape, list):
        full_input_shape = full_input_shape[0]
        
    # 3. Extract features dimension (e.g., 12 from (None, 12))
    if isinstance(full_input_shape, tuple):
        input_dim = full_input_shape[-1] 
    else:
        raise ValueError(f"Could not determine model input shape: {full_input_shape}")

    print(f"\n--- Creating new model with input_shape={input_dim} ---")

    inputs = keras.Input(shape=(input_dim,))
    x = inputs

    # Iterate through original model layers to build the new one
    for i, layer in enumerate(layers[1:]): 
        # Skip non-Dense layers
        if not isinstance(layer, keras.layers.Dense):
            continue 
            
        # Skip the pruned layer
        if i + 1 == target_layer_index:
            print(f"Skipping layer: {layer.name}")
            continue
            
        # Reconstruct layer with original attributes
        x = keras.layers.Dense(layer.units, activation=layer.activation, name=layer.name)(x)
        print(f"Adding layer: {layer.name} with {layer.units} units")

    new_model = keras.Model(inputs=inputs, outputs=x)

    print("\n--- Adjusting weights in the new model ---")
    for layer in new_model.layers:
        if isinstance(layer, keras.layers.Dense):
            # 1. Assign the merged weights to the "next" layer
            if layer.name == next_layer.name:
                layer.set_weights([new_weights, new_bias])
                print(f"✅ Assigned adjusted weights to layer: {layer.name}")
            # 2. Copy original weights for all other layers
            else:
                original_layer = next((c for c in model.layers if c.name == layer.name), None)
                if original_layer is not None:
                    layer.set_weights(original_layer.get_weights())
                    print(f"✅ Copied original weights to layer: {layer.name}")

    # Re-compile model with original configuration
    optimizer = keras.optimizers.Adam(epsilon=1e-8, learning_rate=learning_rate)
    new_model.compile(optimizer=optimizer, loss='mae', metrics=['mse', 'mae'])

    print("\n=== PROCESS COMPLETE ===")
    new_model.summary()

    return new_model

In [ ]:
# OPTION 2 - FORWARD STRATEGY (FS)
def forward_strategy_prune(model, target_layer_index):    
    layers = model.layers

    if target_layer_index < 1 or target_layer_index >= len(layers) - 1:
        raise ValueError("Index must correspond to a hidden layer (not input or output).")

    print("\n=== STARTING PRUNING PROCESS (FS) ===")

    previous_layer = layers[target_layer_index - 1]
    layer_to_prune = layers[target_layer_index]
    next_layer = layers[target_layer_index + 1]

    print("\n--- Layers Involved ---")
    print(f"Previous Layer: {previous_layer.name}")
    print(f"Layer to Prune: {layer_to_prune.name}")
    print(f"Next Layer: {next_layer.name}")

    # Original weights from the next layer
    next_weights, next_bias = next_layer.get_weights()

    print(f"\nLayer to prune: {layer_to_prune.name}")
    print(f"Next layer: {next_layer.name}")
    print(f"Original next layer weights shape: {next_weights.shape}")

    print(f"\nRetrieving weights and bias from target layer: {layer_to_prune.name}")
    print(f"Using next layer '{next_layer.name}' as reference.")

    input_weights, input_bias = layer_to_prune.get_weights()
    print(f"\nTarget layer weights (shape {input_weights.shape}):\n{input_weights}")
    print(f"Target layer bias:\n{input_bias}")

    ref_weights, ref_bias = next_layer.get_weights()
    print(f"\nReference layer weights (shape {ref_weights.shape}):\n{ref_weights}")
    print(f"Reference layer bias:\n{ref_bias}")

    num_neurons_prev = input_weights.shape[0]
    num_neurons_prune = input_weights.shape[1]
    num_neurons_next = next_layer.units

    print("\n--- Weight Adjustment ---")
    print(f"Neurons in previous layer: {num_neurons_prev}")
    print(f"Neurons in layer to prune: {num_neurons_prune}")
    print(f"Neurons in next layer: {num_neurons_next}")

    # --- CORRECTION 1: Determine new input dimension ---
    if hasattr(previous_layer, "units"):
        # If the previous layer is Dense, use its units
        new_input_dim = previous_layer.units
    else:
        # If previous is InputLayer, use model's input shape
        full_input_shape = model.input_shape
        if isinstance(full_input_shape, list):
            full_input_shape = full_input_shape[0]
        # Extract feature dimension (e.g., 12 from (None, 12))
        new_input_dim = full_input_shape[-1] 
    
    output_neurons = next_layer.units

    print(f"\nAdjusting weights for layer '{next_layer.name}' from {next_weights.shape} → ({new_input_dim}, {output_neurons})")

    if next_weights.shape[0] > new_input_dim:
        print("\n🔻 Row reduction via mean similarity")
        original_means = np.mean(next_weights, axis=1)
        # Generate reference means with uniform spacing
        reference_means = np.linspace(np.min(original_means), np.max(original_means), new_input_dim)

        print("\n▶ Original row means:")
        print(original_means)

        print("\n▶ Generated reference means (uniform spacing):")
        print(reference_means)

        selected_indices = []
        used_indices = set()
        for i, ref in enumerate(reference_means):
            distances = np.abs(original_means - ref)
            sorted_order = np.argsort(distances)
            print(f"\n→ Comparing with reference {i} ({ref:.4f}):")
            print("  Distances:", distances)
            print("  Order:", sorted_order)

            for idx in sorted_order:
                if idx not in used_indices:
                    selected_indices.append(idx)
                    used_indices.add(idx)
                    print(f"  ✅ Selected index {idx} (mean {original_means[idx]:.4f})")
                    break

        print("\n✅ Final indices selected to keep:")
        print(selected_indices)

        # Slice the weights using the selected row indices
        next_weights = next_weights[selected_indices, :]

    elif next_weights.shape[0] < new_input_dim:
        print("\n----------- Next layer has more neurons; FS not applicable -----------")
        return model
        
    print(f"\n✅ Final adjusted weights shape: {next_weights.shape}")

    # --- CORRECTION 2: Get input shape for model reconstruction ---
    full_input_shape = model.input_shape
    if isinstance(full_input_shape, list):
        full_input_shape = full_input_shape[0]
    input_dim = full_input_shape[-1] 
    
    print(f"\n--- Creating new model with input_shape={input_dim} ---")

    inputs = keras.Input(shape=(input_dim,))
    x = inputs

    for i, layer in enumerate(layers[1:]):
        if i + 1 == target_layer_index:
            print(f"Skipping pruned layer: {layer.name}")
            continue
        
        # Ensure only Dense layers are processed
        if not isinstance(layer, keras.layers.Dense):
            continue
            
        x = keras.layers.Dense(layer.units, activation=layer.activation, name=layer.name)(x)

    new_model = keras.Model(inputs=inputs, outputs=x)

    # Assign weights to the new model
    for layer in new_model.layers:
        if isinstance(layer, keras.layers.Dense):
            if layer.name == next_layer.name:
                layer.set_weights([next_weights, next_bias])
                print(f"✅ Assigned new weights to layer: {layer.name}")
            else:
                # Copy original weights for unchanged layers
                original_layer = next((c for c in model.layers if c.name == layer.name), None)
                if original_layer:
                    layer.set_weights(original_layer.get_weights())
                    print(f"✅ Copied original weights to layer: {layer.name}")

    # Recompile with the global learning rate
    optimizer = keras.optimizers.Adam(epsilon=1e-8, learning_rate=learning_rate)
    new_model.compile(optimizer=optimizer, loss='mae', metrics=['mse', 'mae'])

    print("\n=== PROCESS COMPLETE ===")
    new_model.summary()

    return new_model